# PL心率主链：先完成PS预热，再初始化视频

原25/25Hz＋5fps、4×3视频运动、MPU融合、HDMI数值/视频及浏览器波形全部保留。仅优化不可变记录转换与归一化；两侧连续性/接触监督仍独立。80ms、500ms、1秒保护、180秒总启动、700点预热、GC及原硬件顺序不变。保留启动摘要减耗；PS完整校验复用同字节摘要与本次嵌套AB结果，缓存关闭清空后才开始原700点预热。第一单元检查实际新PS入口。正式入口先用原工厂完成一次700点纯软件PS预热，再初始化视频并交接同一个PS；不增加第二个子进程或第二次预热。不启用5秒诊断计时。正式60秒，只手动尝试一次；失败保存、不自动重试。

先结束其他采集/计算，连接摄像头、HDMI及传感器。PS退出故障仍单独如实记录，本包不宣称修复。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
from IPython.display import display, FileLink
assert sys.platform == 'linux', '仅在板子上运行'
root = Path.cwd()
assert root.name == 'v3b25 evidence early60 retry3', '请在新的优化目录运行'
from evidence_reuse_guard import verify
checked = verify(root)
print({k: checked[k] for k in ('bundle_validated', 'source_members_unchanged', 'package_members', 'manifest_sha256')})
from run_evidence_reuse60 import short, prepare, run
short_report = short(expected_manifest_sha256=checked['manifest_sha256'])
if (root/'lean-ps-short-results.zip').is_file():
    display(FileLink('lean-ps-short-results.zip'))
assert short_report['passed'], '无硬件短检查未通过，保留结果，不启动硬件'
print('实际PS入口READY秒数：', short_report['ready_info']['warmup_seconds'])
print('预热前摘要缓存已释放：', short_report['ready_info']['ps_validation_prefix']['retained_bytes'] == 0)


In [ ]:
# 仅准备浏览器波形；等绿色就绪后运行下一单元。
prepared = prepare(expected_manifest_sha256=checked['manifest_sha256'])


In [ ]:
assert not globals().get('_evidence_reuse_used', False), '已尝试，保留结果，不重跑'
confirmation = input('其他任务均结束，放稳手指；允许加载原电路并做一次60秒联合实测？输入 RUN-EVIDENCE-REUSE60: ')
assert confirmation == 'RUN-EVIDENCE-REUSE60', '未确认，不启动硬件'
_evidence_reuse_used = True
print('启动总保护180秒、700点预热另计；正式60秒。不要中断、重跑或关闭收尾中的内核。', flush=True)
try:
    report = run(hardware=True, confirmation=confirmation,
                 expected_manifest_sha256=checked['manifest_sha256'])
finally:
    if (root/'evidence-reuse60-results.zip').is_file():
        display(FileLink('evidence-reuse60-results.zip'))
    else:
        print('尚未确认收尾/导出，保留当前输出，不启动另一入口。')
